# Super TikTakToe: lessons about the long game, in self-play on Google Colab

This trains a **new network from scratch**, the size of large (96 channels, 8 blocks), with exactly the settings large had.
Besides the moves and the value it also learns extra lessons about how the game ends. In play it is the same network.
It is compared with large's own log at the same iterations: **14% at iteration 20, 28% at 40, 35% at 60, 43% at 80, 48% at 100** against medium.

There are three networks to train. Each one needs its own Colab session (open this notebook once per session and
set `NETWORK` in the second cell). They do not disturb each other, every one has its own files in the Drive folder.

| `NETWORK` | What it learns besides moves and value |
|---|---|
| `large_long` | final owner of every open small board, the result as win / tie / loss, the search's value of every move |
| `large_long_no_owner` | the same without the final owner of the boards |
| `large_repeat` | nothing extra: large once more, to see how much two equal trainings differ by chance |

1. `alphazero_medium.pt` must be in the Drive folder `Super-TikTakToe-models` (it is).
2. Menu **Runtime > Change runtime type**: choose the L4.
3. Set `NETWORK` and run the cells from top to bottom. Every line is one iteration; every 4th one has the check against medium (256 games).
4. It runs 100 iterations. My guess is 2.5 to 3 hours (5 to 6 compute units) per network; the first lines show the real time.
5. After a disconnect run all cells again: it continues where it stopped.
6. When it has ended: **Runtime > Disconnect and delete runtime**.

In [ ]:
# Which network this session trains: 'large_long', 'large_long_no_owner' or 'large_repeat'
NETWORK = 'large_long'

# Google Drive: the folder with the finished network
from google.colab import drive
drive.mount('/content/drive')
MODELS = '/content/drive/MyDrive/Super-TikTakToe-models'
import os
os.environ['SUPER_TTT_MODELS'] = MODELS

In [ ]:
# The code from GitHub (the newest version every time)
%cd /content
!test -d Super-TikTakToe || git clone https://github.com/kasparhui/Super-TikTakToe.git
%cd /content/Super-TikTakToe
!git pull

In [ ]:
# Which graphics card did this session get?
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# The training. Stop with the stop button of this cell: everything is saved, and running it again continues.
!python -u src/train.py run {NETWORK} 100

In [ ]:
# The checks against medium of all networks that exist so far, next to large's, at the same iterations
import os
import pandas as pd
table = None
for name in ('large', 'large_repeat', 'large_long', 'large_long_no_owner'):
    path = f'{MODELS}/alphazero_{name}_log.csv'
    if not os.path.exists(path):
        continue
    log = pd.read_csv(path)
    log = log[log['vs_medium'].notna() & (log['iteration'] <= 100)][['iteration', 'vs_medium']].rename(columns={'vs_medium': name})
    table = log if table is None else table.merge(log, on='iteration', how='outer')
table.sort_values('iteration').round(3)